<a href="https://colab.research.google.com/github/obaid147/uae-house-price-prediction/blob/main/uae_house_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import pandas as pd

df = pd.read_csv('uae_real_estate_2024.csv')  # match your exact uploaded filename
df.head()

,title,displayAddress,bathrooms,bedrooms,addedOn,type,price,verified,priceDuration,sizeMin,furnishing,description
0,Great ROI I High Floor I Creek View,"Binghatti Canal, Business Bay, Dubai",3,2,2024-08-14T12:02:53Z,Residential for Sale,2500000,True,sell,1323 sqft,NO,MNA Properties is delighted to offer this apar...
1,Full Sea View | Beach Life | Brand New Residence,"La Vie, Jumeirah Beach Residence, Dubai",3,2,2024-08-13T05:45:16Z,Residential for Sale,7200000,True,sell,1410 sqft,YES,"Apartment for sale in La Vie, Jumeirah Beach R..."
2,Green Belt | Corner Unit | Spacious Plot,"La Rosa 6, Villanova, Dubai Land, Dubai",3,4,2024-08-14T06:24:28Z,Residential for Sale,3600000,True,sell,2324 sqft,NO,Treo Homes is very pleased to be bringing to t...
3,2BR+Study | Near Pool and Park | Private,"Springs 15, The Springs, Dubai",3,2,2024-08-15T06:07:22Z,Residential for Sale,2999999,True,sell,1647 sqft,NO,2 Bedrooms + Study | Near Pool &amp; Park | Pr...
4,Vacant | Well Maintained | Area Expert,"Noor Townhouses, Town Square, Dubai",3,3,2024-08-09T08:28:59Z,Residential for Sale,2449999,True,sell,2105 sqft,NO,-Type 1\n-3 Bed+Maid\n-Close To Amenities\n-BU...


In [25]:
# Checking shape and counting null values
print(df.shape)
df.isnull().sum()

(5058, 12)


,0
title,0
displayAddress,0
bathrooms,120
bedrooms,123
addedOn,0
type,0
price,0
verified,0
priceDuration,0
sizeMin,0


In [26]:
print(df['bathrooms'].unique())
print(df['bedrooms'].unique())

['3' '7' '4' '1' '5' '2' '6' nan '7+' 'none']
['2' '4' '3' '5' '1' 'studio' nan '7' '6' '7+']


In [27]:
# Clean bathrooms
df['bathrooms'] = df['bathrooms'].replace({'7+': '7', 'none': '0'})
df['bathrooms'] = pd.to_numeric(df['bathrooms'])

# Clean bedrooms
df['bedrooms'] = df['bedrooms'].replace({'7+': '7', 'studio': '0'})
df['bedrooms'] = pd.to_numeric(df['bedrooms'])

# fill missing values (NaNs) with median
df['bathrooms'] = df['bathrooms'].fillna(df['bathrooms'].median())
df['bedrooms'] = df['bedrooms'].fillna(df['bedrooms'].median())

# Furnishing missing value
df['furnishing'] = df['furnishing'].fillna(df['furnishing'].mode()[0])

df.isnull().sum()

,0
title,0
displayAddress,0
bathrooms,0
bedrooms,0
addedOn,0
type,0
price,0
verified,0
priceDuration,0
sizeMin,0


In [28]:

# Sanity check that the numbers look reasonable
df['sizeMin'] = df['sizeMin'].str.replace(' sqft', '').astype(float)
df['sizeMin'].describe()

,sizeMin
count,5058.000000
mean,2549.670621
std,3746.860842
min,82.000000
25%,802.000000
50%,1419.000000
75%,2799.750000
max,100000.000000


In [29]:
# listings above 10,000 sqft
df[df['sizeMin'] > 10000][['title', 'type', 'sizeMin', 'price']]

,title,type,sizeMin,price
70,Single Row | Near School Plot | Large Layout,Residential for Sale,10172.0,2593860
229,Ellington x Quality | 0% commission | Al ain ...,Residential for Sale,14605.0,40526828
282,Beautiful Brand New | 5 Bedroom | Vacant,Residential for Sale,10013.0,19500000
287,Vacant | Well Maintained | High Number,Residential for Sale,13398.0,53000000
328,"Investment Opportunity, Full Golf course view ...",Residential for Sale,11376.0,9385200
...,...,...,...,...
4873,Private Beach Access | Sea View | New Launch,Residential for Sale,13500.0,44000000
4875,Genuine I Luxury Waterfront Home I Viewing Now!,Residential for Sale,16527.0,23000000
4978,7 Bedroom (G+1) | Spacious | Pool,Residential for Sale,12000.0,11000000
4985,HUGE PLOT I AMAZING LOCATION I LAND INVESTMENT,Residential for Sale,10010.0,2502662


In [30]:
# checking the pattern on how locations are written
df['displayAddress'].head(10)

,displayAddress
0,"Binghatti Canal, Business Bay, Dubai"
1,"La Vie, Jumeirah Beach Residence, Dubai"
2,"La Rosa 6, Villanova, Dubai Land, Dubai"
3,"Springs 15, The Springs, Dubai"
4,"Noor Townhouses, Town Square, Dubai"
5,"The Residences 1, The Residences, Downtown Dub..."
6,"Vida Residences Dubai Marina, Dubai Marina, Dubai"
7,"Anya 2, Arabian Ranches 3, Dubai"
8,"Madinat Hind, Mulberry, Damac Hills 2, Dubai"
9,"East 40, Al Furjan, Dubai"


In [31]:
# The 20 most common areas, which will become one of our key features for predicting price.
df['Area'] = df['displayAddress'].str.split(',').str[-2].str.strip()
df['Area'].value_counts().head(20)

,count
Area,
Downtown Dubai,241
Jumeirah Village Circle,212
Mohammed Bin Rashid City,212
Dubai Marina,187
Business Bay,181
Dubai Hills Estate,157
Palm Jumeirah,145
Al Furjan,122
Dubai Creek Harbour (The Lagoons),117


In [32]:
# Checking for total unique areas
print(df['Area'].nunique())

190


In [33]:
# keep the top N most common areas, and group everything else into "Other"
# 190 is too many to one hot encode directly
top_areas = df['Area'].value_counts().nlargest(50).index
df['Area'] = df['displayAddress'].str.split(',').str[-2].str.strip()
df['Area'] = df['Area'].where(df['Area'].isin(top_areas), 'Other')
df['Area'].value_counts()

# nlargest(20)  grabs the 20 most frequent areas
# where(...) keeps those 20 area names as-is, replaces everything else with "Other"
# Other" is now 983 (~19%), reasonable, with 50 real named areas carrying most of the signal.

,count
Area,
Other,983
Downtown Dubai,241
Mohammed Bin Rashid City,212
Jumeirah Village Circle,212
Dubai Marina,187
Business Bay,181
Dubai Hills Estate,157
Palm Jumeirah,145
Al Furjan,122


In [34]:
# one-hot encode Area, type, and furnishing
df = pd.get_dummies(df, columns=['Area', 'type', 'furnishing'], drop_first=True)
print(df.shape)

(5058, 62)


In [36]:
# drop the columns we don't need for modeling and set up X/y
df = df.drop(columns=['title', 'displayAddress', 'addedOn', 'verified', 'priceDuration', 'description'])

X = df.drop(columns=['price'])
y = df['price']

print(X.shape)
print(y.shape)

"""
title, description — free text, not usable as-is
displayAddress — already extracted into Area, no longer needed
addedOn — listing date, not relevant to price prediction
verified, priceDuration — not useful signals (priceDuration is probably all "sell" anyway)
""";

(5058, 55)
(5058,)


In [37]:
# train/test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)

(4046, 55)
(1012, 55)


In [41]:
# train our first model using linear regression as we are predicting a continuous number
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: {mae:,.0f} AED") # on average, how far off our predictions are,
# MAE of 500,000 means predictions are typically off by about 500k AED.

print(f"R² Score: {r2:.2f}") # how much of the price variation our model explains
# r2_score for Real estate models often land around 0.6-0.85 depending on data quality.

Mean Absolute Error: 3,097,763 AED
R² Score: 0.63


In [42]:
print(y.describe())
"""
huge skew. Median price is 2.35M, but the max goes up to 199 million AED,
with a mean (5M) way above the median (2.35M)
""";

count    5.058000e+03
mean     5.050924e+06
std      1.000928e+07
min      1.000000e+05
25%      1.200000e+06
50%      2.350000e+06
75%      4.499375e+06
max      1.990000e+08
Name: price, dtype: float64


In [43]:
# Handling the price sckew
import numpy as np

# Log-transform the target
y_train_log = np.log(y_train)
y_test_log = np.log(y_test)

# Train on log-transformed price
model_log = LinearRegression()
model_log.fit(X_train, y_train_log)

# Predict (in log scale), then convert back to real price
y_pred_log = model_log.predict(X_test)
y_pred_actual = np.exp(y_pred_log)

mae_log = mean_absolute_error(y_test, y_pred_actual)
r2_log = r2_score(y_test, y_pred_actual)

print(f"Mean Absolute Error: {mae_log:,.0f} AED")
print(f"R² Score: {r2_log:.2f}")

Mean Absolute Error: 1,895,504 AED
R² Score: 0.72


Above improves things noticeably, especially MAE